# HostWise - Track D: Problem and Domain Understanding

**Project:** HostWise - Airbnb nightly price prediction and price positioning
**Cities:** London, Paris, Rome, Madrid, Barcelona (Inside Airbnb data, `hostwise_5cities.csv`)

This notebook defines the problem HostWise addresses, its two tasks, the relevant domain context, the project scope, the hypotheses to test and how success will be judged. Supporting evidence is in Track A (data understanding), Track C (data quality) and Track B (EDA); references such as "C7" point to the relevant section. Statements are marked as **confirmed** (executed output), **reported** (team or earlier analysis), **hypothesis** or **pending** (decision not yet approved).

## 1. Problem Statement

Airbnb nightly prices vary widely between cities, neighbourhoods and listing types, and they are strongly right-skewed within every city (confirmed, C9). A host who wants to set or review a price must answer a comparison question: *what do listings like mine usually ask for in this city?* Comparing manually against a few visible listings is slow and easily biased towards listings that are not really comparable.

HostWise addresses this with two connected capabilities:

1. **Estimate** the nightly price that is typical for a listing with given characteristics.
2. **Position** the listing's advertised price against that estimate as *Underpriced*, *Fairly Priced* or *Overpriced*.

Potential users are new hosts looking for a starting price, existing hosts checking whether their price is far from comparable listings, and small property managers screening several listings.

**Scope of the claim.** HostWise learns from **observed advertised prices**. Its estimate is a *market-typical asking price*, not an objectively fair, optimal or revenue-maximising price. It does not observe bookings, revenue or the price guests finally pay, so it cannot say whether a listing will be booked or how a price change would affect earnings.

## 2. Project Objectives

HostWise has two interconnected tasks. Task 2 is built on the output of Task 1.

| | Task 1 - Nightly price prediction | Task 2 - Price positioning |
|---|---|---|
| Type | Supervised regression (primary ML task) | Rule-based classification derived from Task 1 |
| Input | Listing characteristics: property type and size, location, capacity, amenities, reviews, host attributes | Advertised price $p$ and predicted price $\hat p$ |
| Output | Predicted nightly price in local currency | Underpriced / Fairly Priced / Overpriced |
| Trained model? | Yes - on observed advertised prices (`price`) | No - a fixed rule applied to Task 1 predictions |

**Task 1 - Regression.** Predict a listing's advertised nightly price from characteristics a host knows before setting a price. A log transformation of the target is a candidate because of the strong skew; it will be compared with a raw-price target.

**Task 2 - Rule-based price classification.** Compare the advertised price with the prediction using a relative deviation, for example $d = \ln(p / \hat p)$, and assign a category with a threshold $t$: *Underpriced* if $d < -t$, *Overpriced* if $d > t$, otherwise *Fairly Priced*. No second model is trained, because there are no ground-truth labels showing that a listing is truly under- or overpriced. The threshold is **not yet chosen**; it will be justified against the model's error after Task 1 is evaluated.

**Why the two tasks are linked.** Every prediction error of Task 1 can move a listing into the wrong category, and models tend to under-predict expensive listings and over-predict cheap ones. Task 2 is therefore only as reliable as Task 1, and it must use predictions from a model that did not see the listing's own price (out-of-sample predictions). The categories describe where an advertised price sits relative to the model's estimate - nothing more.

## 3. Domain Understanding

The factors below are expected to relate to nightly asking prices. All relationships in this project are **observational associations**, not causal effects.

- **City and location.** Each city is a separate market with its own price level, regulation and supply structure; within a city, proximity to attractions and transport is usually associated with higher prices.
- **Room type.** Entire homes, private rooms, shared rooms and hotel rooms are different products; less privacy is usually associated with a lower price. Entire homes are 77.0% of active listings (confirmed, A3).
- **Capacity and size.** Guest capacity, bedrooms, beds and bathrooms describe size, but they are strongly related to each other, so their additional information beyond capacity is uncertain. Bathroom privacy (private vs shared) is a separate aspect of the product.
- **Amenities.** Amenities can make a listing more attractive, but they also mark the standard of a property, so an amenity may be associated with price without causing it.
- **Reviews and hosts.** Ratings signal quality but cluster near the maximum (median 4.82, A2). Review counts partly reflect past bookings, which depend on price. Professional multi-listing hosts may price differently from individual hosts.
- **Minimum stay.** Listings requiring 30+ nights are effectively monthly rentals with lower nightly prices - a different product from short stays (confirmed, C7).
- **Advertised vs transaction price.** Guests may pay a different amount because of date-specific pricing, length-of-stay discounts and fees. The data is a snapshot of advertised prices at scrape time [1].
- **Currency.** London prices are in GBP; Paris, Rome, Madrid and Barcelona prices are in EUR (confirmed, C3). Prices are therefore analysed and evaluated per city and never pooled across currencies.

[1] Inside Airbnb, *Data Assumptions*: the data is "a snapshot of listings available at a particular time". https://insideairbnb.com/data-assumptions/

## 4. Dataset & Project Scope

**Cities.** New York was removed because about 78.4% of its listings required a minimum stay of 30 nights or more (reported, six-city Day 1 analysis), which does not fit a short-stay pricing tool. In the five remaining cities most active listings allow stays of up to 29 nights, from 63.4% in Barcelona to 99.0% in London (confirmed, C7).

**Populations.** The full dataset and the modelling population are different and must not be confused (confirmed, A1):

| Population | Definition | Listings |
|---|---|---|
| All listings | Five-city file | 245,402 |
| Active | Listings in the current scrape (`source == "city scrape"`) | 174,922 |
| Priced | Active listings with a price | 168,725 |
| Short-stay scope | Priced listings with `minimum_nights <= 29` | 153,812 |

**Why a short-stay restriction.** Listings with 30+ night minimums have a 22-70% lower median nightly price than short stays in the same city (confirmed, C7). Keeping them would make the model learn two pricing logics. The restriction is used in the exploration but is **pending** team approval for modelling.

**City imbalance.** London (38.2%) and Paris (25.6%) dominate the short-stay population, while Barcelona contributes only 5.3% (confirmed, A1). Results will therefore be reported per city.

**Pending scope decisions:** the 29-night threshold; whether Hotel and Shared rooms (each under 1% of active listings) stay in scope; the price outlier rule (a per-city rule fitted on training data only is proposed, C9).

## 5. Research Questions & Hypotheses

**Research questions**

- **RQ1.** Which listing characteristics are associated with the advertised nightly price within each city?
- **RQ2.** Can a regression model predict advertised nightly prices more accurately than simple baselines in every city?
- **RQ3.** Can the regression model's predictions support an interpretable three-category price-positioning system?

**Hypotheses**

| # | Hypothesis | How it is tested | Status |
|---|---|---|---|
| H1 | Within each city, entire homes are priced higher than private rooms, and price increases with guest capacity | Within-city comparison (Track B, B2-B3) | To be confirmed in EDA |
| H2 | Location explains price variation beyond property size | Price vs distance and neighbourhood at the same size (B6) | To be confirmed in EDA |
| H3 | Bedrooms, bathrooms and amenities add information beyond capacity and room type | Association at the same capacity (B4, B8); feature ablation in modelling | EDA, then model evaluation |
| H4 | The relationships between listing characteristics and price differ across cities | City-level comparison of the above; per-city model errors | Partly supported (market structure differs, C5, C7); to be tested |
| H5 | A regression model beats the strongest simple baseline - the median price per city, room type and capacity - in every city | Held-out test set, per-city metrics | Model evaluation |

RQ3 is answered after Task 1 is evaluated (section 6).

## 6. Success Criteria & Limitations

**Task 1 - Regression**
- Beat simple, transparent baselines (city median; city x room type; city x room type x capacity) in **every** city, on a held-out test set.
- Report MAE in local currency per city (interpretable) and error on the log scale (relative, comparable across currencies), plus an analysis of the largest errors.
- **Pending:** the primary metric, whether cities are weighted equally when comparing models, and a train/test split grouped by host so that listings of the same host are not in both sets. All preprocessing is fitted on training data only.

**Task 2 - Price positioning**
- There are no independent ground-truth labels, so accuracy, precision, recall or F1 **cannot be computed** and will not be reported.
- Instead, the categories are assessed by: justifying the threshold against the model's error per city; checking how category shares change across candidate thresholds (sensitivity); checking how often listings change category when the model is retrained (stability); and checking that categories do not simply mirror price level.

**Main limitations**
- **Asking prices only:** predictions describe typical advertised prices at one point in time, not transaction prices, demand or revenue.
- **Data quality:** missing values, a small number of extreme or implausible prices, and columns that leak the target (excluded from features, C4).
- **Selection:** the model represents active, priced, short-stay listings only and should not be applied beyond that population.
- **City differences and imbalance:** markets differ in structure and size, so performance may vary by city.
- **Unobserved quality:** photos, condition or exact location are not in the data; their effect ends up in the gap between advertised and predicted price.
- **Task 2 depends on Task 1:** model errors and bias pass directly into the categories, which must never be presented as fair market value.

**Next steps:** record the pending Track B results to test H1-H4, agree the pending decisions as a team, then build the preprocessing pipeline and baselines on Day 2.